# Collaborative Filtering based Recommender System using K Nearest Neighbor

- User-based collaborative filtering is based on the user similarity or neighborhood
- Item-based collaborative filtering is based on similarity among items

#### User-based collaborative filtering works.
- User-based collaborative filtering looks for users who are similar. This is very similar to the user clustering method done previously; where we employed explicit user profiles to calculate user similarity. However, the user profiles may not be available, so how can we determine if two users are similar?

#### User-item interaction matrix 
For most collaborative filtering-based recommender systems, the main dataset format is a 2-D matrix called the user-item interaction matrix. In the matrix,  its row is labeled as the user id/index and column labelled to be the item id/index, and the element `(i, j)` represents the rating of user `i` to item `j`.  




#### KNN-based collaborative filtering

Each row vector represents the rating history of a user and each column vector represents the users who rated the item. A user-item interaction matrix is usually very sparse as you can imagine one user very likely only interacts with a very small subset of items and one item is very likely to be interacted by a small subset of users.

Item-based collaborative filtering works similarly, we just need to look at the user-item matrix vertically. Instead of finding similar users, we are trying to find similar items (courses). If two courses are enrolled by two groups of similar users, then we could consider the two items are similar and use the known ratings from the other users to predict the unknown ratings.

If we formulate the KNN based collaborative filtering,  the predicted rating of user $u$ to item $i$, $\hat{r}_{ui}$ is given by:

**User-based** collaborative filtering:
$$\hat{r}_{ui} = \frac{
\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v) \cdot r_{vi}}
{\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v)}$$


**Item-based** collaborative filtering:
$$\hat{r}_{ui} = \frac{
\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j) \cdot r_{uj}}
{\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j)}$$


Here $N^k_i(u)$ notates the nearest k neighbors of $u$.


From the above figure, suppose we want to predict the rating of `user6` to item `Machine Learning Capstone` course. After some similarity measurements, we found that k = 4 nearest neighbors: `user2, user3, user4, user5` with similarities in array ```knn_sims```:


In [1]:
import numpy as np
import math

In [2]:
# Similarity array stores the similarity of user2, user3, user4, and user5 to user6
knn_sims = np.array([0.8, 0.92, 0.75, 0.83])

In [3]:
# Rating on the `Machine Learning Capstone`
knn_ratings = np.array([3.0, 3.0, 2.0, 3.0]) 

In [4]:
# Predicted rating of `user6` to item `Machine Learning Capstone` course 
r_u6_ml =  np.dot(knn_sims, knn_ratings)/ sum(knn_sims)
r_u6_ml

2.7727272727272725

In [5]:
# True rating to be 3.0, then we get a prediction error RMSE (Rooted Mean Squared Error)
true_rating = 3.0
rmse = math.sqrt(true_rating - r_u6_ml) ** 2
rmse

0.22727272727272751

The predicted rating is around 2.7 (close to 3.0 with RMSE 0.22), which indicates that `user6` is also likely to complete the course `Machine Learning Capstone`. As such, we may recommend it to user6 with high confidence.


# Perform KNN-based collaborative filtering on the user-item interaction matrix


In [8]:
import pandas as pd
rating_url = "ratings.csv"
rating_df = pd.read_csv(rating_url)
rating_df.head(2)

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3


In [10]:
# The dataset contains three columns, `user id` (learner), `item id`(course), and `rating`(enrollment mode). 
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head(2)

,user,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
0,2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
1,4,0.0,0.0,0.0,0.0,5.0,3.0,4.0,5.0,3.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,3.0,3.0


The dense format is more preferred as it saves a lot of storage and memory space. While the benefit of the sparse matrix is it is in the nature matrix format and you could apply computations such as cosine similarity directly.

Perform KNN-based collaborative filtering on the user-item interaction matrix. 

You may choose one of the two following implementation options of KNN-based collaborative filtering. 
- The first one is to use `scikit-surprise` which is a popular and easy-to-use Python recommendation system library. 
- The second way is to implement it with standard `numpy`, `pandas`, and `sklearn`. 

You may need to write a lot of low-level implementation code along the way.

## Implementation Option 2: Use `numpy`, `pandas`, and `sklearn`

In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from math import sqrt

# ----------------------
# Load dataset
# ----------------------
df = pd.read_csv("ratings.csv")

# Create user-item interaction matrix
user_item_matrix = df.pivot_table(index='user', columns='item', values='rating')

# Fill NaNs with 0s for similarity computation (or you can use mean-centering)
interaction_filled = user_item_matrix.fillna(0)

# ----------------------
# Calculate user-user similarity
# ----------------------
user_similarity = cosine_similarity(interaction_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=interaction_filled.index, columns=interaction_filled.index)

# ----------------------
# Function to predict rating using KNN
# ----------------------
def predict_rating(user_id, item_id, k=5):
    if item_id not in user_item_matrix.columns:
        return np.nan  # Unknown item

    # Get similarity scores for the user with all other users
    similarities = user_similarity_df[user_id]

    # Get ratings of all users for the target item
    item_ratings = user_item_matrix[item_id]

    # Filter out users who haven't rated the item
    valid_ratings = item_ratings[item_ratings.notna()]

    # Join similarity scores with valid ratings
    neighbors = pd.concat([similarities, valid_ratings], axis=1, join='inner')
    neighbors.columns = ['similarity', 'rating']

    # Sort by similarity
    top_k_neighbors = neighbors.sort_values('similarity', ascending=False).head(k)

    if top_k_neighbors['similarity'].sum() == 0:
        return np.nan  # Avoid divide-by-zero

    # Weighted average of neighbor ratings
    weighted_sum = np.dot(top_k_neighbors['rating'], top_k_neighbors['similarity'])
    sim_sum = top_k_neighbors['similarity'].sum()
    predicted_rating = weighted_sum / sim_sum

    return predicted_rating

# ----------------------
# Prepare test set
# ----------------------
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# Rebuild the user-item matrix only with training data
user_item_matrix = train_df.pivot_table(index='user', columns='item', values='rating')
interaction_filled = user_item_matrix.fillna(0)
user_similarity = cosine_similarity(interaction_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=interaction_filled.index, columns=interaction_filled.index)

# ----------------------
# Predict on test set
# ----------------------
actual_ratings = []
predicted_ratings = []

for _, row in test_df.iterrows():
    user_id = row['user']
    item_id = row['item']
    actual = row['rating']

    # Predict rating
    try:
        pred = predict_rating(user_id, item_id, k=5)
    except:
        pred = np.nan

    if not np.isnan(pred):
        actual_ratings.append(actual)
        predicted_ratings.append(pred)

# ----------------------
# Evaluate with RMSE
# ----------------------
rmse = sqrt(mean_squared_error(actual_ratings, predicted_ratings))
print(f"RMSE: {rmse:.4f}")


RMSE: 0.9610
